In [3]:
import time
from datetime import datetime
from pathlib import Path
import sqlite3
import requests
import pandas as pd
from sub_districts import ensure_carpark_sub_districts

DB_PATH = "parking.db"
CARPARKS_STAMP_FILE = Path(".carparks_last_update")


def load_target_ids() -> set[str]:
    conn = sqlite3.connect(DB_PATH)
    try:
        df = pd.read_sql_query("SELECT park_id FROM carparks", conn)
    finally:
        conn.close()
    return set(df["park_id"].astype(str))


target_ids = load_target_ids()


def update_carparks() -> int:
    """從政府 API 同步全港停車場基本資料（upsert，保留既有 sub_district）。"""
    url = "https://api.data.gov.hk/v1/carpark-info-vacancy"
    params = {"data": "info", "lang": "zh_TW"}
    response = requests.get(url, params=params, timeout=30)
    if response.status_code != 200:
        raise RuntimeError(f"carparks API HTTP {response.status_code}")

    results = response.json().get("results", [])
    if not results:
        raise RuntimeError("carparks API 回傳空結果")

    conn = sqlite3.connect(DB_PATH)
    try:
        cols = {row[1] for row in conn.execute("PRAGMA table_info(carparks)")}
        if "sub_district" not in cols:
            conn.execute("ALTER TABLE carparks ADD COLUMN sub_district TEXT")

        upserted = 0
        for cp in results:
            park_id = str(cp.get("park_Id") or "").strip()
            if not park_id:
                continue
            name = cp.get("name")
            district = cp.get("district")
            latitude = cp.get("latitude")
            longitude = cp.get("longitude")
            address = cp.get("displayAddress")

            cur = conn.execute(
                """
                UPDATE carparks
                SET name = ?, district = ?, latitude = ?, longitude = ?, address = ?
                WHERE park_id = ?
                """,
                (name, district, latitude, longitude, address, park_id),
            )
            if cur.rowcount == 0:
                conn.execute(
                    """
                    INSERT INTO carparks
                    (park_id, name, district, latitude, longitude, address, sub_district)
                    VALUES (?, ?, ?, ?, ?, ?, NULL)
                    """,
                    (park_id, name, district, latitude, longitude, address),
                )
            upserted += 1
        conn.commit()
    finally:
        conn.close()

    ensure_carpark_sub_districts(DB_PATH)
    return upserted


def maybe_update_carparks_monthly() -> None:
    """每個日曆月最多更新一次 carparks。"""
    global target_ids
    now_ym = datetime.now().strftime("%Y-%m")
    if CARPARKS_STAMP_FILE.exists():
        last_ym = CARPARKS_STAMP_FILE.read_text(encoding="utf-8").strip()
        if last_ym == now_ym:
            return

    print(f"開始每月更新 carparks（{now_ym}）...")
    try:
        n = update_carparks()
        CARPARKS_STAMP_FILE.write_text(now_ym, encoding="utf-8")
        target_ids = load_target_ids()
        print(f"carparks 已更新：{n} 筆，目標車場 {len(target_ids)} 個")
    except Exception as e:
        print("carparks 每月更新失敗:", e)


def collect_vacancy():
    url = "https://api.data.gov.hk/v1/carpark-info-vacancy"
    params = {
        "data": "vacancy",
        "lang": "zh_TW",
        "vehicleTypes": "privateCar"
    }
    
    response = requests.get(url, params=params, timeout=30)
    if response.status_code != 200:
        print("請求失敗")
        return
    
    vacancy_results = response.json().get("results", [])
    records = []
    for item in vacancy_results:
        park_id = str(item.get("park_Id"))
        if park_id not in target_ids:
            continue
        private = item.get("privateCar", [])
        if private:
            v = private[0]
            records.append({
                "park_id": park_id,
                "vacancy": v.get("vacancy"),
                "vacancy_type": v.get("vacancy_type"),
                "lastupdate": v.get("lastupdate")
            })
    
    if not records:
        print("沒有新資料")
        return
    
    collected_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    conn = sqlite3.connect("parking.db")
    cursor = conn.cursor()
    
    for r in records:
        cursor.execute("""
            INSERT INTO vacancy_records (park_id, vacancy, vacancy_type, lastupdate, collected_at)
            VALUES (?, ?, ?, ?, ?)
        """, (r["park_id"], r["vacancy"], r["vacancy_type"], r["lastupdate"], collected_at))
    
    conn.commit()
    conn.close()
    print(f"成功收集並寫入 {len(records)} 筆資料 ({collected_at})")
    
def collect_weather():
    weather_url = "https://data.weather.gov.hk/weatherAPI/opendata/weather.php"
    try:
        resp = requests.get(weather_url, params={"dataType": "rhrread", "lang": "en"}, timeout=30)
        weather = resp.json()
        
        hko_temp = None
        for t in weather.get("temperature", {}).get("data", []):
            if "Observatory" in t.get("place", ""):
                hko_temp = t.get("value")
                break
        
        rainfall_list = weather.get("rainfall", {}).get("data", [])
        max_rain = max([r.get("max", 0) for r in rainfall_list]) if rainfall_list else 0
        
        collected_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        conn = sqlite3.connect("parking.db")
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO weather_records (collected_at, temperature, max_rainfall, update_time)
            VALUES (?, ?, ?, ?)
        """, (collected_at, hko_temp, max_rain, weather.get("updateTime")))
        conn.commit()
        conn.close()
        print(f"天氣數據已收集 ({collected_at})")
    except Exception as e:
        print("天氣收集失敗:", e)

if __name__ == "__main__":
    print("開始持續收集（每 10 分鐘一次；carparks 每月更新一次），按 Ctrl+C 停止")
    while True:
        maybe_update_carparks_monthly()
        collect_vacancy()
        collect_weather()
        time.sleep(600)  # 600 秒 = 10 分鐘

開始持續收集（每 10 分鐘一次），按 Ctrl+C 停止
成功收集並寫入 574 筆資料 (2026-10-19 21:13:20)
天氣數據已收集 (2026-10-19 21:13:20)
成功收集並寫入 574 筆資料 (2026-10-19 21:23:20)
天氣數據已收集 (2026-10-19 21:23:21)
成功收集並寫入 574 筆資料 (2026-10-19 21:42:14)
天氣數據已收集 (2026-10-19 21:42:14)
成功收集並寫入 574 筆資料 (2026-10-19 21:52:15)
天氣數據已收集 (2026-10-19 21:52:15)
成功收集並寫入 574 筆資料 (2026-10-19 22:02:15)
天氣數據已收集 (2026-10-19 22:02:16)
成功收集並寫入 574 筆資料 (2026-10-19 22:12:18)
天氣數據已收集 (2026-10-19 22:12:18)
成功收集並寫入 574 筆資料 (2026-10-19 22:22:19)
天氣數據已收集 (2026-10-19 22:22:20)
成功收集並寫入 574 筆資料 (2026-10-19 22:47:42)
天氣數據已收集 (2026-10-19 22:47:42)
成功收集並寫入 574 筆資料 (2026-10-19 22:57:44)
天氣數據已收集 (2026-10-19 22:57:44)
成功收集並寫入 574 筆資料 (2026-10-19 23:07:45)
天氣數據已收集 (2026-10-19 23:07:45)
成功收集並寫入 574 筆資料 (2026-10-19 23:17:46)
天氣數據已收集 (2026-10-19 23:17:46)
成功收集並寫入 574 筆資料 (2026-10-19 23:27:48)
天氣數據已收集 (2026-10-19 23:27:48)
成功收集並寫入 574 筆資料 (2026-10-20 00:18:14)
天氣數據已收集 (2026-10-20 00:18:14)
成功收集並寫入 574 筆資料 (2026-10-20 00:28:15)
天氣數據已收集 (2026-10-20 00:28:15)
成功收集並寫入 574 筆資料 (2

ConnectionError: HTTPSConnectionPool(host='api.data.gov.hk', port=443): Max retries exceeded with url: /v1/carpark-info-vacancy?data=vacancy&lang=zh_TW&vehicleTypes=privateCar (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x174749f90>: Failed to resolve 'api.data.gov.hk' ([Errno 8] nodename nor servname provided, or not known)"))